# 제시 라벨과 일치율

Python 3.12 새 커널에서 교육용 작성자 기준과 judge 예시 라벨부터 확인합니다.

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** 사람 라벨과 judge 라벨을 같은 사례 ID로 맞춘 뒤 정확 일치율과 혼동 행렬을 계산합니다. 혼동 행렬은 실제 라벨 행과 예측 라벨 열의 조합별 개수를 보여 주므로 어떤 등급을 서로 바꾸어 판단했는지 읽을 수 있습니다.

**다른 상황에 적용.** 승인·보류·거절 세 등급을 평가할 때 전체 일치율이 같아도 거절을 승인으로 바꾼 오류와 승인을 보류로 바꾼 오류의 위험은 다릅니다. 등급별 오분류를 따로 보고 운영 임계값을 정합니다.

**비교와 경계.** 이 절의 제시 라벨은 학습용 작성자 기준이며 여러 독립 평가자가 합의한 정답이 아닙니다. 일치율은 라벨 분포가 한쪽으로 치우치면 높아 보일 수 있으므로 사례 수와 등급별 결과를 함께 확인합니다.


## Python 3.12 패키지 버전 고정

**이 셀의 코드 읽기.** 이 셀은 앞 셀에서 만든 값을 입력으로 읽어 위에서 아래 순서로 처리하고, 왼쪽 변수에 다음 셀이 사용할 결과를 저장합니다.


In [1]:
# 실습 환경에서 사용하는 보조 패키지 버전을 고정합니다.
%pip install -q python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 교육용 작성자 기준과 judge 예시 라벨

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [2]:
# 작성자 기준과 judge 예시 라벨 여섯 쌍을 교육용 입력으로 준비합니다.
labels = [
 ('C01', 1, 1), ('C02', 1, 1), ('C03', 0, 0),
 ('C04', 0, 1), ('C05', 1, 1), ('C06', 0, 0),
]
print('라벨 출처: 교육용 작성자 기준(인간 평가 아님)')
print('사례 수:', len(labels))

라벨 출처: 교육용 작성자 기준(인간 평가 아님)
사례 수: 6


## 정확 일치율과 혼동 행렬 계산

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [3]:
# 전체 여섯 사례를 분모로 정확 일치율과 혼동 행렬을 계산합니다.
agree = sum(gold == judge for _, gold, judge in labels) / len(labels)
tp = sum(g == j == 1 for _, g, j in labels)
tn = sum(g == j == 0 for _, g, j in labels)
fp = sum(g == 0 and j == 1 for _, g, j in labels)
fn = sum(g == 1 and j == 0 for _, g, j in labels)
print('정확 일치율:', round(agree, 3))
print({'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn})

정확 일치율: 0.833
{'TP': 3, 'TN': 2, 'FP': 1, 'FN': 0}


## 불일치 사례와 운영 적용 조건 확인

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [4]:
# 사례별 일치 여부를 출력하고 운영에는 독립 검토가 필요함을 표시합니다.
for case_id, author, judge in labels:
    mark = '일치' if author == judge else '불일치'
    print(case_id, author, judge, mark)
assert round(agree, 3) == 0.833
print('운영 적용: 독립적인 도메인 검토자 라벨 필요')

C01 1 1 일치
C02 1 1 일치
C03 0 0 일치
C04 0 1 불일치
C05 1 1 일치
C06 0 0 일치
운영 적용: 독립적인 도메인 검토자 라벨 필요
